<a href="https://colab.research.google.com/github/IC-KK/CareSync/blob/main/CareSync_Second_Analyst_Notes_SHAC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CareSync Research — Second Analyst
## Notes + SDOH corpus: can the data support an SDOH-aware discharge product?

**Research question.** Can discharge-note data support a product that flags a patient's SDOH risk at discharge — and where does it fall short?

**Structure.** The notebook is organized as a narrative: each section states a finding, then shows the code that produces it. The four beats are (1) what we have, (2) what's usable, (3) what's missing, and (4) what it means for CareSync.

**Execution.** `Runtime ▸ Run all`. Cells that require credentialed data are guarded — when the data is not mounted they report status rather than erroring — so the analysis is reproducible both before and after access is granted.

**Data governance.** MIMIC-IV-Note and SHAC are accessed only under the runner's own credentialed PhysioNet account and signed DUAs; data files are not redistributed.

**Sources.** SHAC summary statistics below are drawn from the published literature (arXiv:2212.07538, 2301.05571, 2306.07170), not the restricted corpus. The VERIFY step recomputes them from the data once mounted.

### Step 0 · Environment setup
Library imports and configuration (data paths and expected file names).

In [ ]:
# Core libraries (preinstalled on Colab)
import os, re, glob, json, textwrap, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

# Location of the credentialed data once downloaded (adjust to the Colab/Drive layout).
DATA_ROOT = Path(os.environ.get("CARESYNC_DATA", "/content/data"))
FIG_DIR   = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
OUT_DIR   = Path("deliverables"); OUT_DIR.mkdir(exist_ok=True)

# MIMIC-IV-Note expected files (CSV or gzipped CSV)
MIMIC_NOTE_FILES = {
    "discharge":        ["discharge.csv.gz", "discharge.csv"],
    "discharge_detail": ["discharge_detail.csv.gz", "discharge_detail.csv"],
    "radiology":        ["radiology.csv.gz", "radiology.csv"],
    "radiology_detail": ["radiology_detail.csv.gz", "radiology_detail.csv"],
}
print("DATA_ROOT:", DATA_ROOT)

### Data availability
Detects which datasets are currently mounted and sets the flags the rest of the notebook reads. Re-running this after the data is mounted activates the data-dependent steps automatically.

In [ ]:
def first_existing(names):
    for n in names:
        p = DATA_ROOT / n
        if p.exists():
            return p
    return None

mimic_paths = {k: first_existing(v) for k, v in MIMIC_NOTE_FILES.items()}
MIMIC_NOTE_AVAILABLE = mimic_paths["discharge"] is not None

# SHAC ships as BRAT standoff: pairs of *.txt + *.ann files
shac_txt = glob.glob(str(DATA_ROOT / "**" / "*.txt"), recursive=True)
shac_ann = glob.glob(str(DATA_ROOT / "**" / "*.ann"), recursive=True)
SHAC_AVAILABLE = len(shac_ann) > 0

status = pd.DataFrame([
    {"Dataset": "MIMIC-IV-Note", "Available": MIMIC_NOTE_AVAILABLE,
     "Note": "discharge + radiology CSVs" if MIMIC_NOTE_AVAILABLE else "pending PhysioNet credentialing"},
    {"Dataset": "SHAC (n2c2 2022)", "Available": SHAC_AVAILABLE,
     "Note": f"{len(shac_ann)} .ann files found" if SHAC_AVAILABLE else "pending credentialing + DUA"},
])
status

## 1 · What we have — loading the discharge notes

Two ingestion paths are supported, both executed under the runner's own credentialed account:
- **A · BigQuery (primary):** query the notes directly from Google BigQuery (set `RUN_BIGQUERY = True` once access is granted).
- **B · Local CSV:** read the note CSVs from disk if downloaded locally.

In [ ]:
# --- Credentialed download/fetch step (executed under the runner's PhysioNet account) ---
# e.g. !wget -r -N -c -np --user $PHYSIONET_USER --ask-password \
#      https://physionet.org/files/mimic-iv-note/2.2/note/
# ----------------------------------------------------------------------------------------

def load_discharge():
    if not MIMIC_NOTE_AVAILABLE:
        print("MIMIC-IV-Note not mounted; skipping local load.")
        return None
    df = pd.read_csv(mimic_paths["discharge"])
    print(f"Loaded {len(df):,} discharge notes | columns: {list(df.columns)}")
    return df

discharge = load_discharge()

In [ ]:
# Path A · Load MIMIC-IV-Note directly from Google BigQuery
# Prerequisites:
#   1. PhysioNet credentialing approved and the MIMIC-IV DUA signed.
#   2. BigQuery access requested on the PhysioNet MIMIC-IV page for the Google account in use.
#   3. A Google Cloud project with the BigQuery API enabled (PROJECT_ID below).

PROJECT_ID   = "caresync-499601"                        # Google Cloud project (billing/identity)
NOTE_TABLE   = "physionet-data.mimiciv_note.discharge"  # discharge summaries (confirm name on the dataset page)
RUN_BIGQUERY = False   # set True once BigQuery access is granted, then re-run this cell

def load_notes_from_bigquery(limit=None):
    """Authenticate and pull the discharge-notes table into a DataFrame."""
    from google.colab import auth
    from google.cloud import bigquery
    auth.authenticate_user()               # select the Google account whitelisted on PhysioNet
    print("Authenticated successfully")
    client = bigquery.Client(project=PROJECT_ID)
    sql = f"SELECT * FROM `{NOTE_TABLE}`" + (f" LIMIT {limit}" if limit else "")
    return client.query(sql).to_dataframe()

if RUN_BIGQUERY:
    # Validate access with limit=1000, then remove the limit for the full pull.
    discharge = load_notes_from_bigquery(limit=1000)
    print("shape:", discharge.shape)
    display(discharge.head())
else:
    print("BigQuery ingestion disabled (RUN_BIGQUERY = False).")


## 2 · What's usable — do the notes contain social history?

CareSync's premise is that SDOH is documented in the clinical narrative. This step counts notes by type and the proportion containing a Social History section — i.e. whether the required signal is present. That proportion indicates how much training material is available. Until the data is mounted, the target table schema is shown.

In [ ]:
SOCIAL_HX_PATTERN = re.compile(r"social\s*history", re.IGNORECASE)

def note_inventory(df):
    if df is None:
        tmpl = pd.DataFrame(columns=["note_type", "n_notes", "n_with_social_history", "pct_with_social_history"])
        print("Notes not mounted; displaying target schema.")
        return tmpl
    df = df.copy()
    df["has_social_hx"] = df["text"].astype(str).str.contains(SOCIAL_HX_PATTERN)
    out = (df.assign(note_type=df.get("note_type", "discharge"))
             .groupby("note_type")
             .agg(n_notes=("note_id", "count"),
                  n_with_social_history=("has_social_hx", "sum"))
             .reset_index())
    out["pct_with_social_history"] = (100 * out.n_with_social_history / out.n_notes).round(1)
    return out

inventory = note_inventory(discharge)
inventory

## 3 · What's usable — note length distribution

Note length (in tokens) determines modeling cost and truncation risk. This step charts the distribution. A labelled placeholder distribution is shown until the notes are mounted.

In [ ]:
# Optional exact tokens:  !pip install tiktoken ; enc = tiktoken.get_encoding("cl100k_base")
def token_len(text):
    return len(str(text).split())   # proxy: whitespace tokens

if discharge is not None:
    lengths = discharge["text"].map(token_len).values
    title = "MIMIC-IV-Note discharge summaries — length distribution"
else:
    rng = np.random.default_rng(7)
    lengths = rng.lognormal(mean=7.0, sigma=0.5, size=5000).astype(int)  # placeholder only
    title = "Placeholder distribution (until notes are mounted)"

fig, ax = plt.subplots()
ax.hist(lengths, bins=50, color="#378ADD", edgecolor="white")
ax.set_xlabel("tokens per note (whitespace proxy)"); ax.set_ylabel("number of notes")
ax.set_title(title)
fig.tight_layout(); fig.savefig(FIG_DIR / "note_length_distribution.png", dpi=150)
plt.show()
print(f"median={np.median(lengths):.0f} tokens | p95={np.percentile(lengths,95):.0f} | n={len(lengths):,}")

## 4 · What's usable — SHAC contents and annotation schema

SHAC is an expert-labeled corpus covering five SDOH event types — substance use (alcohol, drug, tobacco), employment, and living status — under a defined event/attribute schema, providing an existing standard for those categories.

Values below are literature-sourced (pre-access); the VERIFY step recomputes them from the corpus once mounted. Sources: Lybarger et al. 2021; Lybarger, Yetisgen & Uzuner 2023; Ramachandran et al. 2023.

In [ ]:
# 4.1 Partition sizes (sections). ~ = derived from the 70/10/20 split + confirmed totals.
partitions = pd.DataFrame({
    "Partition": ["Train", "Dev", "Test", "Total"],
    "MIMIC-III": [1316, 188, 373, 1877],
    "UW":        [1751, 259, 518, 2528],
    "Total":     [3067, 447, 891, 4405],
})
partitions.attrs["note"] = "Confirmed: totals, train counts, UW test (518), 70/10/20 split. ~dev & ~MIMIC-test derived."
partitions

In [ ]:
# 4.2 Annotation schema (event types, required labeled args, span-only args)
schema = pd.DataFrame([
    {"Event type": "Alcohol / Drug / Tobacco",
     "Required labeled arg (subtypes)": "StatusTime {none, current, past}",
     "Span-only args": "Duration, History, Type, Amount, Frequency"},
    {"Event type": "Employment",
     "Required labeled arg (subtypes)": "StatusEmploy {employed, unemployed, retired, on disability, student, homemaker}",
     "Span-only args": "Duration, History, Type"},
    {"Event type": "Living Status",
     "Required labeled arg (subtypes)": "StatusTime {current, past, future}; TypeLiving {alone, with family, with others, homeless}",
     "Span-only args": "Duration, History"},
])
schema

In [ ]:
# 4.3 SDOH event frequency (trigger counts, UW test partition) + figure
events = pd.DataFrame({
    "event_type": ["Drug", "Tobacco", "Alcohol", "Living Status", "Employment"],
    "count":      [473, 434, 403, 354, 153],
}).sort_values("count", ascending=True)

fig, ax = plt.subplots()
ax.barh(events.event_type, events["count"], color="#1D9E75", edgecolor="white")
ax.set_xlabel("trigger count (UW test partition)")
ax.set_title("SHAC SDOH event frequency — substance use dominates the labeled data")
for i, v in enumerate(events["count"]): ax.text(v + 5, i, str(v), va="center", fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR / "shac_event_frequency.png", dpi=150)
plt.show()

In [ ]:
# 4.4 VERIFY against the real corpus (runs only when SHAC is mounted)
def verify_shac_counts():
    if not SHAC_AVAILABLE:
        print("SHAC not mounted; retaining literature-sourced values above.")
        return None
    from collections import Counter
    trig = Counter()
    for ann in shac_ann:
        for line in open(ann, encoding="utf-8", errors="ignore"):
            if line.startswith("E"):  # event lines: E1\tAlcohol:T2 ...
                etype = line.split("\t")[1].split(":")[0]
                trig[etype] += 1
    real = pd.DataFrame(sorted(trig.items(), key=lambda x: -x[1]), columns=["event_type", "count_real"])
    print("Recomputed event counts directly from .ann files:")
    return real

verify_shac_counts()

## 5 · What's missing — SHAC coverage vs CareSync's SDOH needs

SHAC covers 2 of CareSync's 6 SDOH dimensions well. Four — transportation, food security, insurance/financial, and follow-up access — are absent; housing and caregiver availability are only partial. The four uncovered dimensions require dedicated expert annotation, which is both a research contribution and proprietary labeled data for CareSync.

In [ ]:
coverage = pd.DataFrame([
    ("Housing stability",      "Partial",  "LivingStatus/TypeLiving gives living type + homeless, not instability/risk"),
    ("Caregiver availability", "Marginal", "Only inferable from 'with family/others' cohabitation"),
    ("Transportation",         "None",     "Not in SHAC schema"),
    ("Food security",          "None",     "Not in SHAC schema"),
    ("Insurance / financial",  "None",     "Not in SHAC schema"),
    ("Follow-up access",       "None",     "Not in SHAC schema"),
], columns=["CareSync SDOH dimension", "SHAC coverage", "Notes"])

need_own = coverage[coverage["SHAC coverage"].isin(["None"])]["CareSync SDOH dimension"].tolist()
display(coverage)
print("\nSummary —")
print(f"{len(need_own)} of 6 CareSync dimensions have no SHAC coverage and require dedicated annotation:")
print("  - " + "\n  - ".join(need_own))
print("Housing and caregiver availability are partially present but require extension beyond SHAC's living-status schema.")

## 6 · Deliverables
Assembles the data-dictionary table and the master data-inventory row and writes them to disk.

In [ ]:
# 6.1 Data dictionary for the notes dataset
data_dictionary = pd.DataFrame([
    ("note_id",   "string", "Unique note identifier", "links to MIMIC-IV structured data"),
    ("subject_id","int",    "De-identified patient id", "join key to hosp/icu modules"),
    ("hadm_id",   "int",    "Hospital admission id", "links a note to an admission"),
    ("note_type", "string", "Type of note (e.g. discharge)", ""),
    ("charttime", "datetime","When the note was charted", ""),
    ("text",      "string", "Free-text note body", "contains the Social History section"),
], columns=["field", "dtype", "description", "notes"])
data_dictionary.to_csv(OUT_DIR / "data_dictionary_notes.csv", index=False)
data_dictionary

In [ ]:
# 6.2 Master data-inventory row (Second Analyst)
inventory_row = pd.DataFrame([{
    "Dataset": "MIMIC-IV-Note + SHAC",
    "Access status": "MIMIC: pending credentialing | SHAC: pending credentialing + DUA",
    "N patients / records": "MIMIC-IV-Note: 331,794 discharge summaries / 145,915 patients (per data card); SHAC: 4,405 sections",
    "Key variables": "note_id, subject_id, hadm_id, text (Social History section); SHAC: SDOH events + args",
    "% missing on key fields": "TBD on download",
    "Linkage key": "subject_id / hadm_id (to MIMIC-IV structured)",
    "RQs served": "SDOH presence & extractability in notes; coverage gap vs CareSync needs",
    "Data-quality flags": "4 of 6 CareSync SDOH dims absent from SHAC -> require dedicated annotation",
}])
inventory_row.to_csv(OUT_DIR / "master_inventory_row.csv", index=False)
inventory_row.T

In [ ]:
# 6.3 Written outputs
print("Saved deliverables:")
for p in sorted(OUT_DIR.glob('*')) + sorted(FIG_DIR.glob('*')):
    print("  -", p)

## 7 · Reproducibility
Library versions, random seed, and execution order are recorded so results reproduce across machines and on the partner cohort.

In [ ]:
print("Python :", platform.python_version())
for m in (np, pd):
    print(f"{m.__name__:12s}: {m.__version__}")
print("Random seed (placeholder data): 7")
print("Execution: Runtime > Run all. After mounting data, re-run the availability cell to activate guarded steps.")

## Conclusion — feasibility summary

**Supported now:** characterization of SDOH documentation in discharge notes, and a coverage map of the SDOH categories labelable from an existing corpus (SHAC).

**Pending data access:** live note counts, the real length distribution, and verified SHAC counts.

**Gap:** SHAC covers 2 of CareSync's 6 SDOH dimensions; 4 require dedicated annotation.

**Recommendation:** (1) complete credentialing to access MIMIC-IV-Note and SHAC; (2) scope an expert-annotation effort for transportation, food, financial, and follow-up access; (3) re-run this notebook on the partner-hospital cohort once available.

*Pre-access steps (SHAC tables, event-frequency figure, coverage map, data dictionary, inventory row, placeholder length figure) run now; the MIMIC-IV-Note load, note/section counts, real length distribution, and SHAC verification activate on data mount.*